# 99 - Clean up

Removes what bills. Each section asks before deleting.

| Resource | Cost while idle |
| --- | --- |
| Endpoints (staging + prod) | **per hour, continuously** |
| MLflow tracking server | **per hour, continuously** |
| S3 (DVC remote, artifacts, data capture) | per GB-month |
| Training jobs | nothing — already finished |
| IAM role, OIDC provider | nothing |

Delete the endpoints and the MLflow server first; they are the ones that matter.

In [ ]:
import json
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
os.environ["CHURN_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Put this kernel's venv bin/ first on PATH so shell commands (dvc, ...) resolve to the
# venv, however Jupyter was launched.
import sys
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
for line in (PROJECT_ROOT / ".env").read_text().strip().splitlines():
    key, _, value = line.partition("=")
    os.environ[key] = value

from churn.config import Settings, load_params

params = load_params()
settings = Settings.load(params)
print("region:", settings.region, "| bucket:", settings.bucket)

In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name=settings.region)
s3 = boto3.resource("s3", region_name=settings.region)
PROJECT = params["project"]["name"]


def confirm(prompt):
    return input(f"{prompt} [y/N] ").strip().lower() == "y"

## 1. Endpoints

The expensive ones. This lists only endpoints belonging to this project.

In [ ]:
endpoints = [
    ep for ep in sm.list_endpoints(MaxResults=100)["Endpoints"]
    if ep["EndpointName"].startswith(PROJECT)
]
for ep in endpoints:
    print(f"  {ep['EndpointName']:40s} {ep['EndpointStatus']}")
if not endpoints:
    print("  (none)")

In [ ]:
for ep in endpoints:
    name = ep["EndpointName"]
    if not confirm(f"Delete endpoint {name}?"):
        print(f"  kept {name}")
        continue
    config = sm.describe_endpoint(EndpointName=name)["EndpointConfigName"]
    variants = sm.describe_endpoint_config(EndpointConfigName=config)["ProductionVariants"]
    sm.delete_endpoint(EndpointName=name)
    sm.delete_endpoint_config(EndpointConfigName=config)
    for variant in variants:
        try:
            sm.delete_model(ModelName=variant["ModelName"])
        except sm.exceptions.ClientError as exc:
            print(f"    model {variant['ModelName']}: {exc.response['Error']['Message']}")
    print(f"  deleted {name} (+ its config and model)")

## 2. MLflow tracking server

⚠️ This deletes your experiment history and the model registry. The S3 artifact
store survives, but the runs and registered versions do not.

Keep it if you want to look at the metrics later — it is the one resource worth
paying for while you are still iterating.

In [ ]:
servers = sm.list_mlflow_tracking_servers()["TrackingServerSummaries"]
for server in servers:
    print(f"  {server['TrackingServerName']:40s} {server['TrackingServerStatus']}")
if not servers:
    print("  (none)")

In [ ]:
for server in servers:
    name = server["TrackingServerName"]
    if confirm(f"Delete MLflow server {name}? This destroys run history and the registry."):
        sm.delete_mlflow_tracking_server(TrackingServerName=name)
        print(f"  deleting {name}")
    else:
        print(f"  kept {name}")

## 3. Model packages

The SageMaker-side mirror of the MLflow registry. Deleting the MLflow server does
not remove these.

In [ ]:
group = params["project"]["registered_model_name"]
try:
    packages = sm.list_model_packages(ModelPackageGroupName=group, MaxResults=100)[
        "ModelPackageSummaryList"
    ]
    for pkg in packages:
        print(f"  v{pkg['ModelPackageVersion']:<4} {pkg['ModelApprovalStatus']}")
except sm.exceptions.ClientError as exc:
    packages = []
    print("  no such group:", exc.response["Error"]["Message"])

In [ ]:
if packages and confirm(f"Delete all {len(packages)} model packages in {group} and the group itself?"):
    for pkg in packages:
        sm.delete_model_package(ModelPackageName=pkg["ModelPackageArn"])
        print(f"  deleted v{pkg['ModelPackageVersion']}")
    sm.delete_model_package_group(ModelPackageGroupName=group)
    print(f"  deleted group {group}")

## 4. S3

Three prefixes, listed with their sizes so you can decide.

Dropping the **DVC remote** prefix is the destructive one: your committed
pointers will then reference blobs that no longer exist, and `dvc pull` will
fail for everyone. Only do that if you are abandoning the project.

In [ ]:
bucket = s3.Bucket(settings.bucket)
prefixes = {
    "job inputs/outputs": settings.bucket_prefix,
    "DVC remote (data!)": settings.dvc_remote_prefix,
    "MLflow artifacts": "mlflow",
}

for label, prefix in prefixes.items():
    objects = list(bucket.objects.filter(Prefix=prefix))
    size_mb = sum(o.size for o in objects) / 1024**2
    print(f"  {label:22s} s3://{settings.bucket}/{prefix}  {len(objects):5d} objects  {size_mb:8.1f} MiB")

In [ ]:
for label, prefix in prefixes.items():
    warning = "  THIS BREAKS `dvc pull` FOR EVERYONE" if "DVC" in label else ""
    if confirm(f"Delete s3://{settings.bucket}/{prefix}/ ({label})?{warning}"):
        bucket.objects.filter(Prefix=prefix).delete()
        print(f"  deleted {prefix}/")
    else:
        print(f"  kept {prefix}/")

## 5. GitHub Actions IAM

No cost, so only remove these if you are done with the project. Leaving them
means a later `git push` still works.

In [ ]:
ROLE_NAME = "churn-mlops-github-actions"
iam = boto3.client("iam")

try:
    iam.get_role(RoleName=ROLE_NAME)
    if confirm(f"Delete IAM role {ROLE_NAME}?"):
        for policy in iam.list_role_policies(RoleName=ROLE_NAME)["PolicyNames"]:
            iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName=policy)
        iam.delete_role(RoleName=ROLE_NAME)
        print(f"  deleted {ROLE_NAME}")
    else:
        print(f"  kept {ROLE_NAME}")
except iam.exceptions.NoSuchEntityException:
    print(f"  {ROLE_NAME} does not exist")

The OIDC identity provider is shared account-wide — other repos may rely on it.
Leave it unless you know nothing else uses it.

## 6. Local

`dvc gc` prunes the local cache. Note the project's own files (`src`, notebooks,
`params.yaml`, `dvc.lock`) are just git content — nothing to clean.

In [ ]:
import subprocess

print(subprocess.run(["du", "-sh", ".dvc/cache"], capture_output=True, text=True, cwd=PROJECT_ROOT).stdout)
print("Prune cached data not referenced by the current workspace:")
print("  dvc gc --workspace")
print("Also prune on the remote (careful - affects other clones):")
print("  dvc gc --workspace --cloud")

## Verify

Anything still listed here is still billing.

In [ ]:
left_endpoints = [
    ep["EndpointName"] for ep in sm.list_endpoints(MaxResults=100)["Endpoints"]
    if ep["EndpointName"].startswith(PROJECT)
]
left_servers = [
    s["TrackingServerName"] for s in sm.list_mlflow_tracking_servers()["TrackingServerSummaries"]
]

print("endpoints still running:", left_endpoints or "none")
print("MLflow servers still up :", left_servers or "none")

if not left_endpoints and not left_servers:
    print("\nNothing hourly left.")